# WMAPE bottom-up — Sección 1 / tienda 00063 / SKU 127360

## Definición (única fuente de verdad)

1. **Hojas** = `unique_id` con forma `sec||T:tienda||S:sku` (exactamente 2 separadores `||`).
2. Excluir `period_type == "forecast_only"` y filas con `y == 0` o nulo.
3. En cada fila hoja: `abs_err = |y − ŷ|`.
4. Agregar según nivel:

| Nivel | Alcance |
|-------|---------|
| **SKU+tienda** | solo filas de esa hoja |
| **Tienda** | todas las hojas de esa tienda |
| **Sección** | todas las hojas de la sección |

$$\mathrm{wMAPE} = \frac{\sum \mathrm{abs\_err}}{\sum |y|}$$

Tienda y sección **no** usan el `yhat` del nodo agregado RLS. Solo errores de hojas.

Script CLI equivalente: `python artifacts/validate_wmape_bottom_up.py`

In [6]:
from __future__ import annotations

import sys
from pathlib import Path

import polars as pl

PROJECT_ROOT = Path.cwd()
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "settings.py").exists() or (candidate / "app" / "backend.py").exists():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import settings
from app import backend

FORECAST_PATH = Path(settings.FORECAST_PATH)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("FORECAST_PATH:", FORECAST_PATH)
print("exists       :", FORECAST_PATH.exists())

PROJECT_ROOT : d:\Documents\GitHub\tienda-inglesa-v3
FORECAST_PATH: d:\Documents\GitHub\tienda-inglesa-v3\data\output\forecast.parquet
exists       : True


In [7]:
# Parámetros del caso
SECCION = "1"
# STORE = "00063"
# SKU = "127360"
STORE = "00001"
SKU = "46499"

UNIDAD = "Valor ($)"  # default dashboard; cambiar a "Unidades" si hace falta
IN_SAMPLE_ONLY = False  # True → solo period_type == in_sample

res_df = backend.load_forecast_parquet(FORECAST_PATH)
has_value = "value" in res_df.columns and "valuehat" in res_df.columns
unit_df = backend.prepare_unit_df(res_df, UNIDAD, has_value)

if IN_SAMPLE_ONLY and "period_type" in unit_df.columns:
    unit_df = unit_df.filter(pl.col("period_type") == "in_sample")

UID_SECCION = SECCION
UID_TIENDA = settings.make_unique_id(SECCION, store=STORE)
UID_HOJA = settings.make_unique_id(SECCION, store=STORE, sku=SKU)

print(f"unidad={UNIDAD}  in_sample_only={IN_SAMPLE_ONLY}  has_value={has_value}")
print(f"hoja   : {UID_HOJA}")
print(f"tienda : {UID_TIENDA}")
print(f"sección: {UID_SECCION}")
print(f"unit_df: {unit_df.shape}")
if "period_type" in unit_df.columns:
    print("period_type:", unit_df["period_type"].value_counts().sort("period_type"))

unidad=Valor ($)  in_sample_only=False  has_value=True
hoja   : 1||T:00001||S:46499
tienda : 1||T:00001
sección: 1
unit_df: (27131620, 23)
period_type: shape: (3, 2)
┌───────────────┬──────────┐
│ period_type   ┆ count    │
│ ---           ┆ ---      │
│ str           ┆ u32      │
╞═══════════════╪══════════╡
│ forecast_only ┆ 1467750  │
│ in_sample     ┆ 24912441 │
│ out_sample    ┆ 751429   │
└───────────────┴──────────┘


In [8]:
unit_df.filter(pl.col("unique_id")=="1||T:00001||S:46499").write_excel("data_46499.xlsx")

## 1. Hojas scorables

Misma lógica que `backend._scored_leaves` / `validate_wmape_bottom_up._scored_leaves_manual`.

In [9]:
def scored_leaves(df: pl.DataFrame) -> pl.DataFrame:
    leaves = df.filter(
        pl.col("unique_id").str.count_matches(r"\|\|", literal=False) == 2
    )
    if "period_type" in leaves.columns:
        leaves = leaves.filter(pl.col("period_type") != "forecast_only")
    leaves = leaves.filter(
        pl.col("y").is_not_null()
        & pl.col("yhat").is_not_null()
        & (pl.col("y") != 0)
    )
    if leaves.height == 0:
        return leaves
    return leaves.with_columns(
        (pl.col("y") - pl.col("yhat")).abs().alias("abs_err"),
        pl.col("unique_id").str.replace(r"\|\|S:.*$", "").alias("store_uid"),
        pl.col("unique_id").str.split("||").list.get(0).alias("seccion"),
        pl.col("unique_id").str.extract(r"\|\|S:([^|]+)", 1).alias("sku"),
    )


def wmape_of(sub: pl.DataFrame) -> dict:
    if sub.height == 0:
        return {"n_rows": 0, "n_leaves": 0, "sum_y": 0.0, "sum_abs_err": 0.0, "wmape": 0.0}
    sum_y = float(sub["y"].sum())
    sum_err = float(sub["abs_err"].sum())
    return {
        "n_rows": int(sub.height),
        "n_leaves": int(sub["unique_id"].n_unique()),
        "sum_y": sum_y,
        "sum_abs_err": sum_err,
        "wmape": (sum_err / abs(sum_y)) if sum_y else 0.0,
    }


def serie_agregada_wmape(df: pl.DataFrame, uid: str) -> float | None:
    """Método viejo (incorrecto para tienda/sección)."""
    sub = df.filter(pl.col("unique_id") == uid)
    if "period_type" in sub.columns:
        sub = sub.filter(pl.col("period_type") != "forecast_only")
    sub = sub.filter(pl.col("y").is_not_null() & (pl.col("y") != 0))
    if sub.height == 0:
        return None
    sy = float(sub["y"].sum())
    if sy == 0:
        return 0.0
    return float((sub["y"] - sub["yhat"]).abs().sum()) / abs(sy)


leaves = scored_leaves(unit_df)
print(f"Hojas scorables: {leaves.height} filas | {leaves['unique_id'].n_unique() if leaves.height else 0} series")
cols = [c for c in ["unique_id", "ds", "period_type", "y", "yhat", "abs_err", "store_uid", "seccion", "sku"] if c in leaves.columns]
leaves.select(cols).head(5)

Hojas scorables: 5329962 filas | 37144 series


unique_id,ds,period_type,y,yhat,abs_err,store_uid,seccion,sku
str,date,str,f64,f64,f64,str,str,str
"""1||T:00411||S:100942""",2024-06-01,"""in_sample""",168.3,0.95,167.35,"""1||T:00411""","""1""","""100942"""
"""1||T:00411||S:100942""",2024-06-04,"""in_sample""",187.0,1.41,185.59,"""1||T:00411""","""1""","""100942"""
"""1||T:00411||S:100942""",2024-06-05,"""in_sample""",166.78,2.51,164.27,"""1||T:00411""","""1""","""100942"""
"""1||T:00411||S:100942""",2024-06-18,"""in_sample""",187.0,1.28,185.72,"""1||T:00411""","""1""","""100942"""
"""1||T:00411||S:100942""",2024-07-06,"""in_sample""",187.0,1.19,185.81,"""1||T:00411""","""1""","""100942"""


## 2. SKU+tienda `1||T:00063||S:127360`

`wMAPE = Σ abs_err / Σ |y|` solo sobre filas de esa hoja.
En hojas, bottom-up ≡ serie del propio nodo.

In [10]:
hoja = leaves.filter(pl.col("unique_id") == UID_HOJA)
m_hoja = wmape_of(hoja)
old_hoja = serie_agregada_wmape(unit_df, UID_HOJA)

print(f"filas          : {m_hoja['n_rows']}")
print(f"Σ |y|          : {m_hoja['sum_y']:,.6f}")
print(f"Σ abs_err      : {m_hoja['sum_abs_err']:,.6f}")
print(f"wMAPE hoja     : {m_hoja['wmape'] * 100:.4f}%")
if old_hoja is not None:
    print(f"serie del nodo : {old_hoja * 100:.4f}%  (debe coincidir)")

show = [c for c in ["ds", "period_type", "y", "yhat", "abs_err"] if c in hoja.columns]
if hoja.height:
    display(hoja.select(show).head(10))
    display(hoja.select(show).tail(5))

filas          : 697
Σ |y|          : 4,055,551.220000
Σ abs_err      : 823,978.030000
wMAPE hoja     : 20.3173%
serie del nodo : 20.3173%  (debe coincidir)


ds,period_type,y,yhat,abs_err
date,str,f64,f64,f64
2024-05-26,"""in_sample""",8304.76,8304.76,0.0
2024-05-27,"""in_sample""",7575.47,8972.06,1396.59
2024-05-28,"""in_sample""",8146.44,10044.87,1898.43
2024-05-29,"""in_sample""",6897.0,8221.93,1324.93
2024-05-30,"""in_sample""",5935.89,7202.67,1266.78
2024-05-31,"""in_sample""",4901.34,7041.48,2140.14
2024-06-01,"""in_sample""",8044.38,7496.23,548.15
2024-06-02,"""in_sample""",10147.73,7844.86,2302.87
2024-06-03,"""in_sample""",7746.76,8542.71,795.95


ds,period_type,y,yhat,abs_err
date,str,f64,f64,f64
2026-04-22,"""out_sample""",5074.7,5335.28,260.58
2026-04-23,"""out_sample""",5062.69,4803.36,259.33
2026-04-24,"""out_sample""",3459.5,4722.6,1263.1
2026-04-25,"""out_sample""",5793.16,4928.9,864.26
2026-04-26,"""out_sample""",7585.05,5237.45,2347.6


## 3. Tienda `1||T:00063` — bottom-up

Sumar `abs_err` y `y` de **todas** las hojas con `store_uid == 1||T:00063`.
No se usa la serie agregada de la tienda.

In [11]:
ti_leaves = leaves.filter(pl.col("store_uid") == UID_TIENDA)
m_tienda = wmape_of(ti_leaves)
old_tienda = serie_agregada_wmape(unit_df, UID_TIENDA)

print(f"n hojas distintas : {m_tienda['n_leaves']}")
print(f"filas hoja        : {m_tienda['n_rows']}")
print(f"Σ |y|             : {m_tienda['sum_y']:,.6f}")
print(f"Σ abs_err         : {m_tienda['sum_abs_err']:,.6f}")
print(f"wMAPE tienda BU   : {m_tienda['wmape'] * 100:.4f}%")
if old_tienda is not None:
    print(f"(método viejo serie agregada) : {old_tienda * 100:.4f}%  ← NO usar")

# Top hojas de la tienda por contribución a abs_err
if ti_leaves.height:
    contrib = (
        ti_leaves.group_by("unique_id")
        .agg(
            pl.col("y").sum().alias("sum_y"),
            pl.col("abs_err").sum().alias("sum_abs_err"),
        )
        .with_columns(
            (pl.col("sum_abs_err") / pl.col("sum_y").abs()).alias("wmape_hoja")
        )
        .sort("sum_abs_err", descending=True)
    )
    print("\nTop 10 hojas de la tienda por Σabs_err:")
    display(contrib.head(10))

n hojas distintas : 4778
filas hoja        : 1084663
Σ |y|             : 827,291,139.161900
Σ abs_err         : 518,818,321.284900
wMAPE tienda BU   : 62.7129%
(método viejo serie agregada) : 12.0831%  ← NO usar



Top 10 hojas de la tienda por Σabs_err:


unique_id,sum_y,sum_abs_err,wmape_hoja
str,f64,f64,f64
"""1||T:00001||S:12845""",8.2229e6,7.6203e6,0.926714
"""1||T:00001||S:567332""",1.9308e7,4.1531e6,0.215094
"""1||T:00001||S:411262""",4.4884e6,3.9939e6,0.889818
"""1||T:00001||S:46713""",9.3623e6,3.6195e6,0.386604
"""1||T:00001||S:116741""",8.400153e6,3.6065e6,0.429338
"""1||T:00001||S:473932""",7.5303e6,3.6006e6,0.478143
"""1||T:00001||S:483046""",1.0087e7,3.5279e6,0.349731
"""1||T:00001||S:39472""",1.5235e7,3459657.4,0.227085
"""1||T:00001||S:106356""",9.3997e6,3.2968e6,0.350734


## 4. Sección `1` — bottom-up

Misma lógica: Σ abs_err / Σ |y| sobre **todas** las hojas de la sección.

In [12]:
sec_leaves = leaves.filter(pl.col("seccion") == UID_SECCION)
m_sec = wmape_of(sec_leaves)
old_sec = serie_agregada_wmape(unit_df, UID_SECCION)

print(f"n hojas distintas : {m_sec['n_leaves']}")
print(f"filas hoja        : {m_sec['n_rows']}")
print(f"Σ |y|             : {m_sec['sum_y']:,.6f}")
print(f"Σ abs_err         : {m_sec['sum_abs_err']:,.6f}")
print(f"wMAPE sección BU  : {m_sec['wmape'] * 100:.4f}%")
if old_sec is not None:
    print(f"(método viejo serie agregada) : {old_sec * 100:.4f}%  ← NO usar")

n hojas distintas : 23383
filas hoja        : 3536131
Σ |y|             : 1,563,639,883.931900
Σ abs_err         : 1,063,391,305.194900
wMAPE sección BU  : 68.0074%
(método viejo serie agregada) : 9.0957%  ← NO usar


## 5. Resumen manual vs `backend.wmape_bottom_up`

Deben coincidir al floating-point.

In [13]:
manual = pl.DataFrame(
    {
        "nivel": ["sku+tienda", "tienda", "seccion"],
        "unique_id": [UID_HOJA, UID_TIENDA, UID_SECCION],
        "wMAPE_manual_%": [
            round(m_hoja["wmape"] * 100, 4),
            round(m_tienda["wmape"] * 100, 4),
            round(m_sec["wmape"] * 100, 4),
        ],
        "sum_y": [m_hoja["sum_y"], m_tienda["sum_y"], m_sec["sum_y"]],
        "sum_abs_err": [m_hoja["sum_abs_err"], m_tienda["sum_abs_err"], m_sec["sum_abs_err"]],
        "n_rows": [m_hoja["n_rows"], m_tienda["n_rows"], m_sec["n_rows"]],
        "n_leaves": [m_hoja["n_leaves"], m_tienda["n_leaves"], m_sec["n_leaves"]],
        "serie_agregada_%": [
            None if old_hoja is None else round(old_hoja * 100, 4),
            None if old_tienda is None else round(old_tienda * 100, 4),
            None if old_sec is None else round(old_sec * 100, 4),
        ],
    }
)
display(manual)

bu = backend.wmape_bottom_up(unit_df)
check = (
    bu.filter(pl.col("unique_id").is_in([UID_HOJA, UID_TIENDA, UID_SECCION]))
    .with_columns((pl.col("wmape") * 100).round(4).alias("wMAPE_backend_%"))
    .select(["unique_id", "wMAPE_backend_%", "sum_y", "n_with_sales"])
)
display(check)

# Match numérico
bu_map = {r["unique_id"]: r["wmape"] for r in bu.to_dicts()}
for nivel, uid, m in [
    ("sku+tienda", UID_HOJA, m_hoja),
    ("tienda", UID_TIENDA, m_tienda),
    ("seccion", UID_SECCION, m_sec),
]:
    b = bu_map.get(uid)
    if b is None:
        print(f"✗ {nivel}: backend no devolvió {uid}")
        continue
    delta = m["wmape"] - b
    ok = abs(delta) < 1e-12
    print(f"{'✓' if ok else '✗'} {nivel}: manual={m['wmape']*100:.6f}%  backend={b*100:.6f}%  Δ={delta*100:+.6e} pp")

nivel,unique_id,wMAPE_manual_%,sum_y,sum_abs_err,n_rows,n_leaves,serie_agregada_%
str,str,f64,f64,f64,i64,i64,f64
"""sku+tienda""","""1||T:00001||S:46499""",20.3173,4.0556e6,823978.03,697,1,20.3173
"""tienda""","""1||T:00001""",62.7129,8.2729e8,5.1882e8,1084663,4778,12.0831
"""seccion""","""1""",68.0074,1.5636e9,1.0634e9,3536131,23383,9.0957


unique_id,wMAPE_backend_%,sum_y,n_with_sales
str,f64,f64,u32
"""1||T:00001||S:46499""",20.3173,4.0556e6,697
"""1||T:00001""",62.7129,8.2729e8,1084663
"""1""",68.0074,1.5636e9,3536131


✓ sku+tienda: manual=20.317288%  backend=20.317288%  Δ=+0.000000e+00 pp
✓ tienda: manual=62.712907%  backend=62.712907%  Δ=-4.662937e-13 pp
✓ seccion: manual=68.007430%  backend=68.007430%  Δ=-2.897682e-12 pp


## 6. Checks estructurales + `wmape_por_id`

In [14]:
checks = []

# Anidamiento de sumas
checks.append(("Σ|y| hoja ≤ Σ|y| tienda", m_hoja["sum_y"] <= m_tienda["sum_y"] + 1e-6))
checks.append(("Σabs_err hoja ≤ Σabs_err tienda", m_hoja["sum_abs_err"] <= m_tienda["sum_abs_err"] + 1e-6))
checks.append(("Σ|y| tienda ≤ Σ|y| sección", m_tienda["sum_y"] <= m_sec["sum_y"] + 1e-6))
checks.append(("Σabs_err tienda ≤ Σabs_err sección", m_tienda["sum_abs_err"] <= m_sec["sum_abs_err"] + 1e-6))

# wmape_por_id == bottom_up
tabla_ids = backend.wmape_por_id([UID_HOJA, UID_TIENDA, UID_SECCION], unit_df, fill_missing=False)
for uid in (UID_HOJA, UID_TIENDA, UID_SECCION):
    row = tabla_ids.filter(pl.col("unique_id") == uid)
    if row.height == 0 or uid not in bu_map:
        checks.append((f"wmape_por_id tiene {uid}", False))
    else:
        checks.append(
            (
                f"wmape_por_id == bottom_up ({uid})",
                abs(float(row["wmape"][0]) - bu_map[uid]) < 1e-12,
            )
        )

# En hoja, serie agregada == bottom-up
if old_hoja is not None:
    checks.append(("hoja: serie == bottom-up", abs(old_hoja - m_hoja["wmape"]) < 1e-12))

all_ok = True
for name, passed in checks:
    print(f"  [{'✓' if passed else '✗'}] {name}")
    all_ok = all_ok and passed

print()
print("RESULTADO:", "OK" if all_ok else "FAIL")

  [✓] Σ|y| hoja ≤ Σ|y| tienda
  [✓] Σabs_err hoja ≤ Σabs_err tienda
  [✓] Σ|y| tienda ≤ Σ|y| sección
  [✓] Σabs_err tienda ≤ Σabs_err sección
  [✓] wmape_por_id == bottom_up (1||T:00001||S:46499)
  [✓] wmape_por_id == bottom_up (1||T:00001)
  [✓] wmape_por_id == bottom_up (1)
  [✓] hoja: serie == bottom-up

RESULTADO: OK


## 7. Solo in-sample (opcional)

Mismo método restringido a `period_type == "in_sample"`.

In [15]:
if "period_type" not in unit_df.columns:
    print("Sin columna period_type — no aplica.")
else:
    unit_in = backend.prepare_unit_df(res_df, UNIDAD, has_value).filter(
        pl.col("period_type") == "in_sample"
    )
    leaves_in = scored_leaves(unit_in)
    m_h_in = wmape_of(leaves_in.filter(pl.col("unique_id") == UID_HOJA))
    m_t_in = wmape_of(leaves_in.filter(pl.col("store_uid") == UID_TIENDA))
    m_s_in = wmape_of(leaves_in.filter(pl.col("seccion") == UID_SECCION))
    display(
        pl.DataFrame(
            {
                "nivel": ["sku+tienda", "tienda", "seccion"],
                "wMAPE_in_sample_%": [
                    round(m_h_in["wmape"] * 100, 4),
                    round(m_t_in["wmape"] * 100, 4),
                    round(m_s_in["wmape"] * 100, 4),
                ],
                "n_rows": [m_h_in["n_rows"], m_t_in["n_rows"], m_s_in["n_rows"]],
                "sum_y": [m_h_in["sum_y"], m_t_in["sum_y"], m_s_in["sum_y"]],
            }
        )
    )
    tin = backend.wmape_bottom_up(unit_in)
    display(
        tin.filter(pl.col("unique_id").is_in([UID_HOJA, UID_TIENDA, UID_SECCION]))
        .with_columns((pl.col("wmape") * 100).round(4).alias("wMAPE_backend_in_%"))
        .select(["unique_id", "wMAPE_backend_in_%", "sum_y", "n_with_sales"])
    )

nivel,wMAPE_in_sample_%,n_rows,sum_y
str,f64,i64,f64
"""sku+tienda""",20.3643,669,3.9079e6
"""tienda""",62.7018,1042904,7.9480e8
"""seccion""",67.9929,3403973,1.5057e9


unique_id,wMAPE_backend_in_%,sum_y,n_with_sales
str,f64,f64,u32
"""1||T:00001||S:46499""",20.3643,3.9079e6,669
"""1||T:00001""",62.7018,7.9480e8,1042904
"""1""",67.9929,1.5057e9,3403973


## 8. Solo in-sample (opcional)

Mismo método restringido a `period_type == "out_sample"`.

In [16]:
if "period_type" not in unit_df.columns:
    print("Sin columna period_type — no aplica.")
else:
    unit_in = backend.prepare_unit_df(res_df, UNIDAD, has_value).filter(
        pl.col("period_type") == "out_sample"
    )
    leaves_in = scored_leaves(unit_in)
    m_h_in = wmape_of(leaves_in.filter(pl.col("unique_id") == UID_HOJA))
    m_t_in = wmape_of(leaves_in.filter(pl.col("store_uid") == UID_TIENDA))
    m_s_in = wmape_of(leaves_in.filter(pl.col("seccion") == UID_SECCION))
    display(
        pl.DataFrame(
            {
                "nivel": ["sku+tienda", "tienda", "seccion"],
                "wMAPE_in_sample_%": [
                    round(m_h_in["wmape"] * 100, 4),
                    round(m_t_in["wmape"] * 100, 4),
                    round(m_s_in["wmape"] * 100, 4),
                ],
                "n_rows": [m_h_in["n_rows"], m_t_in["n_rows"], m_s_in["n_rows"]],
                "sum_y": [m_h_in["sum_y"], m_t_in["sum_y"], m_s_in["sum_y"]],
            }
        )
    )
    tin = backend.wmape_bottom_up(unit_in)
    display(
        tin.filter(pl.col("unique_id").is_in([UID_HOJA, UID_TIENDA, UID_SECCION]))
        .with_columns((pl.col("wmape") * 100).round(4).alias("wMAPE_backend_in_%"))
        .select(["unique_id", "wMAPE_backend_in_%", "sum_y", "n_with_sales"])
    )

nivel,wMAPE_in_sample_%,n_rows,sum_y
str,f64,i64,f64
"""sku+tienda""",19.0736,28,147620.74
"""tienda""",62.9856,41759,3.2492e7
"""seccion""",68.3844,132158,5.7905e7


unique_id,wMAPE_backend_in_%,sum_y,n_with_sales
str,f64,f64,u32
"""1||T:00001||S:46499""",19.0736,147620.74,28
"""1||T:00001""",62.9856,3.2492e7,41759
"""1""",68.3844,5.7905e7,132158


## Notas

- Ranking del dashboard y `metrics.parquet` usan `wmape_bottom_up` / `wmape_por_id`.
- Tras cambios de código: `python -m app.dashboard_artifacts` y recargar Streamlit.
- Validación CLI: `python artifacts/validate_wmape_bottom_up.py --seccion 1 --store 00063 --sku 127360`
- Unidad por defecto del ranking: **Valor ($)** (`value` / `valuehat`).